In [1]:
cd /mnt/prj-vol/

/__modal/volumes/vo-lwZWPDBHpTCl13LS9ZlseR


In [2]:
# !git clone https://github.com/namwasinyourheart/asr_ft.git

In [3]:
cd asr_ft

/__modal/volumes/vo-lwZWPDBHpTCl13LS9ZlseR/asr_ft


In [4]:
pwd

'/__modal/volumes/vo-lwZWPDBHpTCl13LS9ZlseR/asr_ft'

In [5]:
ls

__pycache__/        eval.py      notebooks/        summarize_results.py
configs/            examples/    prepare_data.py   transcribe.py
download_data.py    exps/        requirements.txt  wandb/
download_models.py  finetune.py  src/


In [42]:
!git remote -v

origin	https://github.com/namwasinyourheart/asr_ft.git (fetch)
origin	https://github.com/namwasinyourheart/asr_ft.git (push)


In [8]:
import os
os.environ['HF_HOME'] = "/mnt/models-vol/cache_huggingface"

In [9]:
!echo $HF_HOME

/mnt/models-vol/cache_huggingface


In [10]:
!apt-get install ffmpeg -y
%uv pip install -r requirements.txt




ffmpeg is already the newest version (7:5.1.7-0+deb12u1).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.
Using Python 3.12.6 environment at: /usr/local
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transformers (HEAD)
Updating https://github.com/huggingface/transfor

In [11]:
ls configs/

03092025/        phowhisper-base.yaml                whisper-base.yaml
28082025/        quan_whisper-base__vietbud500.yaml  whisper-base_ft.yaml
eraxai_wow.yaml  transcribe.yaml


In [12]:
ls configs/03092025

openai_whisper-large-v3-turbo__lsvsc.yaml
openai_whisper-large-v3-turbo__vimd.yaml


In [13]:
from src.utils.exp_utils import print_cfg

In [14]:
print_cfg("configs/28082025/openai_whisper-large-v3-turbo__lsvsc.yaml")

exp_manager:
  prj_name: vnpost_asr_ft
  exp_name: openai_whisper-large-v3-turbo__lsvsc
  exp_variant: wo_ft
  exp_notes: 'data preprocess: filter outaudio, labels with length exceed; donot preprocess
    text for training'
  seed: 202508
  task_name: vi_asr
  dataset_name: null
  model_name: openai/whisper-large-v3-turbo
  phase_name: eval
  print_cfg: true
  exps_dir: exps
  print_model: true
  print_processor: false
  print_trainable_parameters: true
  print_parameter_datatypes: true
  print_peft_config: true
  print_device: true
  print_training_args: false
  wandb:
    project: vnpost_asr_ft
    log_artifact: false
data:
  is_prepared: true
  prepared_data_dir: null
  root_data_dir: /mnt/data-vol/vnpost-asr/LSVSC
  streaming: false
  columns_to_retain:
  - sample_id
  - audio
  - text
  - filename
  subset_ratio: 1
  do_save: true
  do_show: false
  prepared_data_dirname: prepared_data
model:
  model_type: ASR
  pretrained_model_name_or_path: openai/whisper-large-v3-turbo
  pretra

In [15]:
import numpy as np

def wer_corpus_and_macro(refs, hyps, return_details=False):
    def wer_details(ref, hyp):
        r = ref.split()
        h = hyp.split()
        d = np.zeros((len(r)+1, len(h)+1), dtype=np.uint8)

        for i in range(len(r)+1):
            d[i][0] = i
        for j in range(len(h)+1):
            d[0][j] = j

        for i in range(1, len(r)+1):
            for j in range(1, len(h)+1):
                if r[i-1] == h[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    substitute = d[i-1][j-1] + 1
                    insert    = d[i][j-1] + 1
                    delete    = d[i-1][j] + 1
                    d[i][j] = min(substitute, insert, delete)

        i, j = len(r), len(h)
        S = D = I = 0
        while i > 0 or j > 0:
            if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] and r[i-1] == h[j-1]:
                i -= 1
                j -= 1
            elif i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + 1:
                S += 1
                i -= 1
                j -= 1
            elif j > 0 and d[i][j] == d[i][j-1] + 1:
                I += 1
                j -= 1
            else:
                D += 1
                i -= 1

        N = max(1, len(r))
        wer_value = (S + D + I) / N
        return wer_value, S, D, I, N

    total_S = total_D = total_I = total_N = 0
    wer_list = []
    details = []

    for ref, hyp in zip(refs, hyps):
        wer_val, S, D, I, N = wer_details(ref, hyp)
        total_S += S
        total_D += D
        total_I += I
        total_N += N
        wer_list.append(wer_val)

        if return_details:
            details.append({
                "ref": ref,
                "hyp": hyp,
                "wer": wer_val,
                "S": S,
                "D": D,
                "I": I,
                "N": N,
            })

    micro_wer = (total_S + total_D + total_I) / total_N
    macro_wer = float(np.mean(wer_list)) if wer_list else 0.0

    summary = {
        "micro_wer": micro_wer,
        "macro_wer": macro_wer,
        "S": total_S,
        "D": total_D,
        "I": total_I,
        "n_samples": len(refs),
    }
    if return_details:
        return summary, details
    else:
        return summary





In [16]:
# Demo
hyps = ["hãy subscribe cho kênh ghiền mì gõ để không bỏ lỡ những video hấp dẫn"]
refs = ["đầu năm cũng đã cùng với bên khuyến nông khuyến lâm kết hợp bên ủy ban đã chỉ đạo để bên khuyến nông khuyến lâm xuống tuyên truyền bà con nhân dân"]

summary, details = wer_corpus_and_macro(refs, hyps, return_details=True)

print("== Summary ==")
print(summary)

print("\n== Per-sample details ==")
for d in details:
    print(d)

== Summary ==
{'micro_wer': 0.96875, 'macro_wer': 0.96875, 'S': 14, 'D': 17, 'I': 0, 'n_samples': 1}

== Per-sample details ==
{'ref': 'đầu năm cũng đã cùng với bên khuyến nông khuyến lâm kết hợp bên ủy ban đã chỉ đạo để bên khuyến nông khuyến lâm xuống tuyên truyền bà con nhân dân', 'hyp': 'hãy subscribe cho kênh ghiền mì gõ để không bỏ lỡ những video hấp dẫn', 'wer': 0.96875, 'S': 14, 'D': 17, 'I': 0, 'N': 32}


In [17]:
len(refs[0].split())

32

In [18]:
details

[{'ref': 'đầu năm cũng đã cùng với bên khuyến nông khuyến lâm kết hợp bên ủy ban đã chỉ đạo để bên khuyến nông khuyến lâm xuống tuyên truyền bà con nhân dân',
  'hyp': 'hãy subscribe cho kênh ghiền mì gõ để không bỏ lỡ những video hấp dẫn',
  'wer': 0.96875,
  'S': 14,
  'D': 17,
  'I': 0,
  'N': 32}]

In [19]:
%%writefile eval.py
import os

import warnings

import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from torch.utils.data import DataLoader

from hydra import initialize, compose
from omegaconf import OmegaConf

from transformers import set_seed
from src.utils.model_utils import load_whisper_model, load_processor

from src.utils.exp_utils import setup_environment, create_exp_dir

from prepare_data import prepare_data, preprocess_text

from tqdm.auto import tqdm
from evaluate import load
from collections import defaultdict


warnings.filterwarnings("ignore")

import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        batch["filename"] = [f["filename"] for f in features]
        batch["sample_id"] = [f["sample_id"] for f in features]

        return batch


def parse_args():
    import argparse
    parser = argparse.ArgumentParser(description="Load generation config.")
    parser.add_argument("--config_path", type=str, required=True, help="Path to the YAML config file for generating.")

    args, override_args = parser.parse_known_args()
    return args, override_args


def load_cfg(config_path, override_args=None):

    """
    Load a configuration file using Hydra and OmegaConf.
    
    Args:
        config_path (str): Path to the configuration file.
        override_args (list, optional): List of arguments to override configuration values.

    Returns:
        cfg: Loaded configuration object.
    """

    override_args = override_args or []
    config_path = os.path.normpath(config_path)
    
    if not os.path.isfile(config_path):
        raise FileNotFoundError(f"Configuration file not found at: {config_path}")
    
    config_dir = os.path.dirname(config_path)
    config_fn = os.path.splitext(os.path.basename(config_path))[0]
    
    try:
        with initialize(version_base=None, config_path=config_dir):
            cfg = compose(config_name=config_fn, overrides=override_args)
    except Exception as e:
        raise RuntimeError(f"Failed to load configuration from {config_path}: {e}")
    
    exp_args = cfg.exp_manager
    data_args = cfg.data
    # tokenizer_args = cfg.tokenizer
    # prompt_args = cfg.prompt
    model_args = cfg.model
    train_args = cfg.train
    eval_args = cfg.evaluate
    device_args = cfg.device
    gen_args = cfg.generate

    return cfg, exp_args, data_args, model_args, train_args, eval_args, gen_args, device_args

def save_cfg(cfg, config_path):
    """
    Save the configuration to a YAML file.

    Args:
        cfg (OmegaConf): The configuration object to save.
        config_path (str): The path where the configuration file will be saved.

    Returns:
        None
    """
    OmegaConf.save(cfg, config_path)
    print(f"Configuration saved to {config_path}")


import json
import csv
import os


def write_to_txt(file_path, predictions_list):
    """Writes prediction results to a TXT file."""
    with open(file_path, "w", encoding="utf-8") as f:

        for prediction in predictions_list:
            for key, value in prediction.items():
                f.write(f"{key}: {value}\n")
                f.write("-" * 24 + "\n")
            # f.write("-" * 48 + "\n\n")
            f.write("\n\n")


def save_predictions(predictions_list, directory, filename):
    """
    Saves predictions in a format determined by the file extension.

    Args:
        predictions_list (list): List of prediction results.
        directory (str): Directory path to save files.
        filename (str): Filename with extension (e.g., 'results.txt', 'results.json', 'results.csv').
    """

    # Ensure directory exists
    os.makedirs(directory, exist_ok=True)

    # Extract file extension
    file_extension = filename.split('.')[-1].lower()
    file_path = os.path.join(directory, filename)

    # Choose appropriate write function
    if file_extension == "txt":
        write_to_txt(file_path, predictions_list)
    # elif file_extension == "json":
    #     write_to_json(file_path, predictions_list)
    # elif file_extension == "csv":
    #     write_to_csv(file_path, predictions_list)
    else:
        raise ValueError("Unsupported file extension. Use '.txt', '.json', or '.csv'.")

def save_metrics(metrics, directory, filename):
    """
    Saves evaluation metrics (e.g., accuracy) to a TXT file.

    Args:
        metrics (dict): Dictionary containing evaluation metrics.
        directory (str): Directory path to save the file.
        filename (str):  Filename with extension .txt
    """
    file_path = os.path.join(directory, filename)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Experiment Name: {metrics.get('exp_name', 'N/A')}\n")
        f.write(f"Experiment Variant: {metrics.get('exp_variant', 'N/A')}\n")
        f.write("-" * 48 + "\n\n")
        
        for key, value in metrics.items():
            if key != "exp_name" and key != "exp_variant":  # Avoid duplicating experiment name
                f.write(f"{key}: {value}\n")
        
        f.write("-" * 48 + "\n")

import pandas as pd 
def summarize_metric(metric_by_group: dict, 
                     model_name: str = "model_1", 
                     top_n_province: int = 5,
                     filename: str = "summary_metrics.csv"
                     ):
    row = {"Model": model_name}
    
    # Dialect
    for k, v in metric_by_group["dialect"].items():
        row[f"Dialect_{k}"] = v
    
    # Gender
    gender_map = {1: "Male", 0: "Female", "male": "Male", "female": "Female"}
    for k, v in metric_by_group.get("gender", {}).items():
        row[f"Gender_{gender_map.get(k, str(k))}"] = v

    
    # Province: Get top-n that having highest WER
    for prov, val in sorted(
        metric_by_group.get("province_name", {}).items(),
        key=lambda x: -x[1]
    )[:top_n_province]:
        row[f"Province_{prov}"] = val

    df = pd.DataFrame([row])
    df.to_csv(filename, index=False)
    
    return df
import numpy as np


def calculate_wer_per_sample(ref, hyp):
        r = ref.split()
        h = hyp.split()
        d = np.zeros((len(r)+1, len(h)+1), dtype=np.uint8)

        for i in range(len(r)+1):
            d[i][0] = i
        for j in range(len(h)+1):
            d[0][j] = j

        for i in range(1, len(r)+1):
            for j in range(1, len(h)+1):
                if r[i-1] == h[j-1]:
                    d[i][j] = d[i-1][j-1]
                else:
                    substitute = d[i-1][j-1] + 1
                    insert    = d[i][j-1] + 1
                    delete    = d[i-1][j] + 1
                    d[i][j] = min(substitute, insert, delete)

        i, j = len(r), len(h)
        S = D = I = 0
        while i > 0 or j > 0:
            if i > 0 and j > 0 and d[i][j] == d[i-1][j-1] and r[i-1] == h[j-1]:
                i -= 1
                j -= 1
            elif i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + 1:
                S += 1
                i -= 1
                j -= 1
            elif j > 0 and d[i][j] == d[i][j-1] + 1:
                I += 1
                j -= 1
            else:
                D += 1
                i -= 1

        N = max(1, len(r))
        wer_value = (S + D + I) / N
        return wer_value, S, D, I, N

def calculate_wer(refs, hyps, return_details=False):

    total_S = total_D = total_I = total_N = 0
    wer_list = []
    details = []

    for ref, hyp in zip(refs, hyps):
        wer_val, S, D, I, N = calculate_wer_per_sample(ref, hyp)
        total_S += S
        total_D += D
        total_I += I
        total_N += N
        wer_list.append(wer_val)

        if return_details:
            details.append({
                "ref": ref,
                "hyp": hyp,
                "wer": wer_val,
                "S": S,
                "D": D,
                "I": I,
                "N": N,
            })

    micro_wer = (total_S + total_D + total_I) / total_N
    macro_wer = float(np.mean(wer_list)) if wer_list else 0.0

    summary = {
        "micro_wer": micro_wer,
        "macro_wer": macro_wer,
        "S": total_S,
        "D": total_D,
        "I": total_I,
        "N": total_N,
        "n_samples": len(refs),
    }
    if return_details:
        return summary, details
    else:
        return summary

def main():
    setup_environment()

    # Parse arguments
    args, override_args = parse_args()

    # Load configuration
    cfg, exp_args, data_args, model_args, train_args, eval_args, gen_args, device_args = load_cfg(args.config_path, override_args)


    if cfg.exp_manager.print_cfg:
        print(OmegaConf.to_yaml(cfg))


    # Create experiment directories
    exp_name = cfg.exp_manager.exp_name
    exps_dir = cfg.exp_manager.exps_dir
    exp_variant = cfg.exp_manager.exp_variant

    (exp_dir, exp_variant_dir, exp_variant_data_dir, exp_variant_checkpoints_dir, exp_variant_results_dir) = create_exp_dir(exp_name, exp_variant, exps_dir)

    # Save configuration if have any changes from the overrides
    config_path = os.path.join(exp_variant_dir, f'{exp_name}__{exp_variant}.yaml')
    save_cfg(cfg, config_path)

    # Set seed
    set_seed(exp_args.seed)

    # Get dataset
    dataset, all_sid2meta = prepare_data(exp_args, data_args, model_args, device_args)

    
    # Load model and processor
    from transcribe import load_model_for_transcribe
    
    model = load_model_for_transcribe(model_args, device_args)
    # model.generation_config.language = "vi"
    # model.generation_config.task = "transcribe"
    # model.generation_config.forced_decoder_ids = None

    from accelerate import Accelerator
    accelerator = Accelerator(cpu=device_args.use_cpu)

    model.eval()
    model = model.to(accelerator.device)
    
    processor = load_processor(model_args)
    tokenizer = processor.tokenizer
    
    test_ds = dataset['test']

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )
    test_dataloader = DataLoader(test_ds, 
                                 batch_size=eval_args.batch_size, 
                                 collate_fn=data_collator
                                )
    # wer_metric = load("wer")
    predictions_list = []
    all_preds, all_refs = [], []
    
    grouped_preds = {
        "dialect": defaultdict(list),
        "province_name": defaultdict(list),
        "gender": defaultdict(list),
    }
    grouped_labels = {
        "dialect": defaultdict(list),
        "province_name": defaultdict(list),
        "gender": defaultdict(list),
    }
    
    
    for step, batch in enumerate(tqdm(test_dataloader, desc="Evaluating...")):

        if step == eval_args.break_step:
                break
        
        with torch.no_grad():
            input_features = batch["input_features"].to(model.device, dtype=model.dtype)
            
            generated_tokens = model.generate(
                input_features=input_features,
                return_dict_in_generate=True,
                max_new_tokens=gen_args.max_new_tokens,
            ).sequences.cpu().numpy()

            # print("[DEBUG] generated_tokens.shape:", generated_tokens.shape)
            
            labels = batch["labels"].cpu().numpy()
            labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
            decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

            predictions = [preprocess_text(p) for p in decoded_preds]
            ground_truth = [preprocess_text(g) for g in decoded_labels]


            # wer_metric.add_batch(predictions=predictions, 
            #                      references=ground_truth)

            all_preds.extend(predictions)
            all_refs.extend(ground_truth)

            for i, sid in enumerate(batch["sample_id"]):
                # fn = batch["filename"][i]
                pred = predictions[i]
                label = ground_truth[i]
                meta = all_sid2meta['test'][str(sid)]
                sample_wer, S, D, I, N = calculate_wer_per_sample(label, pred)

                 # Append prediction info
                predictions_list.append({
                    "sid": sid,
                    # "filename": fn,
                    "meta": meta,
                    "prediction": pred,
                    "label": label,
                    "wer": sample_wer,
                    "S": S,
                    "D": D,
                    "I": I,
                    "N": N
                })
            
    
            for i, sid in enumerate(batch["sample_id"]):
                meta = all_sid2meta['test'][str(sid)]
                for key in grouped_preds.keys():

                    if key not in meta:   # <-- skip missing entirely
                        continue
                    
                    grouped_preds[key][meta[key]].append(decoded_preds[i])
                    grouped_labels[key][meta[key]].append(decoded_labels[i])
    
    # Calculate WER
    # micro_wer = wer_metric.compute()
    metrics_wer = calculate_wer(all_refs, all_preds, return_details=False)
            
    # Save predictions
    save_predictions(predictions_list, 
                     exp_variant_results_dir, 
                     eval_args.prediction_filename,
                    )
    # Compute WER
    # print("Micro WER:", micro_wer)
    print("Metrics WER:", 100 * metrics_wer["micro_wer"])

    wer_by_group = {}

    for meta_key in grouped_preds.keys():
        wer_by_group[meta_key] = {}
        for group_value, preds in grouped_preds[meta_key].items():
            refs = grouped_labels[meta_key][group_value]
            if len(preds) == 0 or len(refs) == 0:
                continue  # skip empty groups
            metric_tmp = load("wer")
            metric_tmp.add_batch(predictions=preds, references=refs)
            wer_by_group[meta_key][group_value] = 100 * metric_tmp.compute()

        # if the whole meta_key is missing → drop it
        if not wer_by_group[meta_key]:
            wer_by_group.pop(meta_key)


    print("WER by Group:", wer_by_group)

    # Save metrics
    metrics = {
        "exp_name": exp_args.exp_name,
        "exp_variant": exp_args.exp_variant,
        "micro_wer": metrics_wer["micro_wer"],
        "macro_wer": metrics_wer["macro_wer"],
        "S": metrics_wer["S"],
        "D": metrics_wer["D"],
        "I": metrics_wer["I"],
        "N": metrics_wer["N"],
        "wer_by_group": wer_by_group,
        "n_samples": metrics_wer["n_samples"],
    }

    save_metrics(metrics, 
                 exp_variant_results_dir, 
                 eval_args.metric_filename)

    summarize_metric(wer_by_group,
                     model_name=exp_args.exp_name + '_' + exp_args.exp_variant,
                     top_n_province=10,
                     filename=os.path.join(exp_variant_results_dir, f"summary_metrics_{exp_args.exp_name}_{exp_args.exp_variant}.csv")
                     )
    
if __name__ == "__main__":
    main()

Overwriting eval.py


In [20]:
data.prepared_data_dir="/mnt/data-vol/vnpost-asr/ViMD/exps/openai_whisper-large-v3-turbo__vimd__label_text_normalized" \

SyntaxError: incomplete input (1106400704.py, line 1)

In [21]:
%%writefile configs/03092025/openai_whisper-large-v3-turbo__lsvsc.yaml
exp_manager:
  prj_name: vnpost_asr_ft
  exp_name: openai_whisper-large-v3-turbo__lsvsc
  exp_variant: wo_ft
  exp_notes: 
  seed: 202508
  task_name: vi_asr
  dataset_name: null
  model_name: openai/whisper-large-v3-turbo
  phase_name: eval
  print_cfg: true
  exps_dir: exps
  print_model: true
  print_processor: false
  print_trainable_parameters: true
  print_parameter_datatypes: true
  print_peft_config: true
  print_device: true
  print_training_args: false
  wandb:
    project: vnpost_asr_ft
    log_artifact: false
data:
  is_prepared: true
  prepared_data_dir: null
  root_data_dir: /mnt/data-vol/vnpost-asr/LSVSC
  streaming: false
  columns_to_retain:
  - sample_id
  - audio
  - text
  - filename
  subset_ratio: 1
  do_save: true
  do_show: false
  prepared_data_dirname: prepared_data
model:
  model_type: ASR
  pretrained_model_name_or_path: openai/whisper-large-v3-turbo
  pretrained_processor_name_or_path: null
  load_in_4bit: false
  load_in_8bit: false
  bnb_4bit_compute_dtype: null
  bnb_4bit_quant_type: nf4
  bnb_4bit_use_double_quant: false
  bnb_4bit_quant_storage: uint8
  torch_dtype: float16
  attn_implementation: null
  device_map: null
  low_cpu_mem_usage: null
  adapter_path: null
train:
  use_peft: false
  lora:
    r: 32
    lora_alpha: 64
    lora_dropout: 0.0
    bias: none
    task_type: null
    inference_mode: false
    target_modules:
    - q_proj
    - k_proj
    - v_proj
    - o_proj
    - gate_proj
    - up_proj
    - down_proj
    modules_to_save: []
  merge_after_train: true
  train_n_samples: -1
  val_n_samples: 128
  test_n_samples: 128
  do_resume_from_checkpoint: false
  train_metric_filename: train_metrics.json
  eval_metrics:
  - wer
  train_args:
    _target_: transformers.Seq2SeqTrainingArguments
    resume_from_checkpoint: null
    do_train: true
    do_eval: true
    do_predict: false
    learning_rate: 5.0e-05
    warmup_steps: 500
    num_train_epochs: 1
    per_device_train_batch_size: 128
    per_device_eval_batch_size: 192
    logging_steps: 1
    logging_first_step: true
    save_strategy: epoch
    eval_strategy: steps
    eval_steps: 10
    eval_accumulation_steps: 1
    eval_on_start: true
    use_cpu: false
    report_to: None
    remove_unused_columns: false
generate:
  max_new_tokens: 256
  pad_token_id: null
  skip_special_tokens: true
  temperature: null
  do_sample: null
evaluate:
  batch_size: 128
  break_step: -1
  do_extract_prediction: true
  metric_filename: test_metrics.txt
  prediction_filename: test_predictions.txt
  result_filename: test_result.txt
device:
  use_cpu: false

Overwriting configs/03092025/openai_whisper-large-v3-turbo__lsvsc.yaml


In [30]:
ls /mnt/prj-vol/asr_ft/exps/openai_whisper-large-v3-turbo__vimd/label_text_normalized/checkpoints

all_results.json  checkpoint-29/  eval_results.json   trainer_state.json
checkpoint-0/     checkpoint-3/   train_results.json


In [32]:
ls /mnt/prj-vol/asr_ft/exps/openai_whisper-large-v3-turbo__vimd/label_text_normalized/results/adapter

README.md  adapter_config.json  adapter_model.safetensors


In [ ]:
%%writefile configs/03092025/openai_whisper-large-v3-turbo__vimd.yaml
exp_manager:
  prj_name: vnpost_asr_ft
  exp_name: "openai_whisper-large-v3-turbo__vimd"
  exp_variant: "label_text_normalized"
  exp_notes: "label_text normalized for training"
  seed: 202508
  task_name: 'vi_asr'
  dataset_name: 
  model_name: 'openai/whisper-large-v3-turbo'
  phase_name: 'eval'
  print_cfg: true
  exps_dir: 'exps'
  print_model: true
  print_processor: false
  print_trainable_parameters: true
  print_parameter_datatypes: true
  print_peft_config: true
  print_device: true
  print_training_args: false
  wandb:
    project: vnpost_asr_ft
    log_artifact: false

data:
  is_prepared: true 
  prepared_data_dir: 
  root_data_dir: "/mnt/data-vol/vnpost-asr/ViMD"
  streaming: false
  columns_to_retain: ["sample_id", "audio", "text", "filename"]
  subset_ratio: 1
  # do_split: true
  # val_ratio: 0.25
  # test_ratio: 0.2
  do_save: true
  do_show: false
  prepared_data_dirname: prepared_data

model: 
  model_type: "ASR"
  pretrained_model_name_or_path: "openai/whisper-large-v3-turbo"
  pretrained_processor_name_or_path: 
  load_in_4bit: false
  load_in_8bit: false
  bnb_4bit_compute_dtype: null
  bnb_4bit_quant_type: "nf4"
  bnb_4bit_use_double_quant: false
  bnb_4bit_quant_storage: "uint8"
  torch_dtype: float16
  attn_implementation: null
  device_map: null
  low_cpu_mem_usage: null
  adapter_path: 

train:
    use_peft: true
    lora:
      r: 32
      lora_alpha: 64
      lora_dropout: 0.0
      bias: none
      task_type: 
      inference_mode: false
      target_modules: ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
      modules_to_save: []


    merge_after_train: true
    train_n_samples: -1
    val_n_samples: 128
    test_n_samples: 128

    do_resume_from_checkpoint: false

    train_metric_filename: 'train_metrics.json'
    eval_metrics: ['wer']

    train_args:
      _target_: transformers.Seq2SeqTrainingArguments
      resume_from_checkpoint: 
      do_train: true
      do_eval: true
      do_predict: false
      learning_rate: 1e-6
      # warmup_steps: 500
      warmup_ratio: 0.1
      num_train_epochs: 1
      # max_steps: 1
      per_device_train_batch_size: 128
      # gradient_accumulation_steps=1
      per_device_eval_batch_size: 192
      # logging_strategy: "no"
      logging_steps: 1
      logging_first_step: true
      save_strategy: steps
      save_steps: 100
      eval_strategy: steps
      eval_steps: 10
      eval_accumulation_steps: 1
      eval_on_start: true
      use_cpu: false
      report_to: None
      remove_unused_columns: false

generate:
  max_new_tokens: 448
  num_beams: 1
  condition_on_prev_tokens: False
  compression_ratio_threshold: 1.35 
  temperature: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  return_timestamps: True


evaluate:
  batch_size: 128
  break_step: -1
  metric_filename: 'test_metrics.txt'
  prediction_filename: 'test_predictions.txt'
  result_filename: 'test_result.txt'

device:
  use_cpu: False

In [ ]:
!python eval.py  \
    --config_path=configs/03092025/openai_whisper-large-v3-turbo__vimd.yaml \
    exp_manager.exp_variant="label_text_normalized" \
    evaluate.break_step=1 \
    evaluate.batch_size=16


In [39]:
!python eval.py  \
    --config_path=configs/03092025/openai_whisper-large-v3-turbo__vimd.yaml \
    exp_manager.exp_variant="label_text_normalized" \
    model.adapter_path="/mnt/prj-vol/asr_ft/exps/openai_whisper-large-v3-turbo__vimd/label_text_normalized/checkpoints/checkpoint-29/" \
    evaluate.break_step=1 \
    evaluate.batch_size=16


exp_manager:
  prj_name: vnpost_asr_ft
  exp_name: openai_whisper-large-v3-turbo__vimd
  exp_variant: label_text_normalized
  exp_notes: label_text normalized for training
  seed: 202508
  task_name: vi_asr
  dataset_name: null
  model_name: openai/whisper-large-v3-turbo
  phase_name: eval
  print_cfg: true
  exps_dir: exps
  print_model: true
  print_processor: false
  print_trainable_parameters: true
  print_parameter_datatypes: true
  print_peft_config: true
  print_device: true
  print_training_args: false
  wandb:
    project: vnpost_asr_ft
    log_artifact: false
data:
  is_prepared: true
  prepared_data_dir: null
  root_data_dir: /mnt/data-vol/vnpost-asr/ViMD
  streaming: false
  columns_to_retain:
  - sample_id
  - audio
  - text
  - filename
  subset_ratio: 1
  do_save: true
  do_show: false
  prepared_data_dirname: prepared_data
model:
  model_type: ASR
  pretrained_model_name_or_path: openai/whisper-large-v3-turbo
  pretrained_processor_name_or_path: null
  load_in_4bit: fal

In [23]:
!python summarize_results.py

Results saved to exps/results/results_summary.csv


In [26]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.read_csv('exps/results/results_summary.csv')

,Experiment Name,Experiment Variant,Results File,Overall WER,dialect_northern,dialect_highland_central,dialect_None,dialect_central,dialect_southern,dialect_minority_ethnic_group,dialect_North,dialect_Central,dialect_South,gender_female,gender_male,gender_None,gender_1,gender_0,province_name_CaoBang,province_name_LangSon,province_name_QuangNinh,province_name_HaiPhong,province_name_ThaiBinh,province_name_NamDinh,province_name_PhuTho,province_name_ThaiNguyen,province_name_YenBai,province_name_TuyenQuang,province_name_HaGiang,province_name_LaoCai,province_name_LaiChau,province_name_SonLa,province_name_DienBien,province_name_HoaBinh,province_name_HaNoi,province_name_HaiDuong,province_name_NinhBinh,province_name_ThanhHoa,province_name_NgheAn,province_name_HaTinh,province_name_DaNang,province_name_DakLak,province_name_DakNong,province_name_LamDong,province_name_HoChiMinh,province_name_DongNai,province_name_BinhDuong,province_name_LongAn,province_name_TienGiang,province_name_VinhLong,province_name_CanTho,province_name_DongThap,province_name_AnGiang,province_name_KienGiang,province_name_CaMau,province_name_TayNinh,province_name_BenTre,province_name_BaRiaVungTau,province_name_QuangBinh,province_name_QuangTri,province_name_ThuaThienHue,province_name_QuangNgai,province_name_BinhDinh,province_name_PhuYen,province_name_KhanhHoa,province_name_GiaLai,province_name_KonTum,province_name_SocTrang,province_name_TraVinh,province_name_NinhThuan,province_name_BinhThuan,province_name_VinhPhuc,province_name_HungYen,province_name_HaNam,province_name_QuangNam,province_name_BinhPhuoc,province_name_BacLieu,province_name_HauGiang,province_name_BacKan,province_name_BacGiang,province_name_BacNinh
0,openai_whisper-large-v3-turbo__lsvsc,label_text_normalized,test_metrics.txt,0.141994,28.870673,23.943662,30.322581,29.304093,30.751174,31.343284,NaN,NaN,NaN,27.629511,30.486318,31.533646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,openai_whisper-large-v3-turbo__vimd,label_text_normalized,test_metrics.txt,0.180646,NaN,NaN,NaN,NaN,NaN,NaN,25.899978,31.847483,27.188388,NaN,NaN,NaN,29.484239,24.616283,26.511628,26.041144,33.544612,28.89991,24.453941,23.904382,26.736943,25.929598,25.428571,25.02439,28.502879,24.518201,23.179134,26.05042,23.114224,23.341772,20.883745,24.290579,25.962488,22.663682,32.880845,37.844144,26.603001,30.632184,25.539957,28.498845,29.201486,23.652837,19.972325,24.56715,28.271484,32.726269,39.120757,24.985233,27.389603,24.555336,24.731183,27.067669,24.3818,25.203252,40.579135,32.876712,40.051414,39.864865,34.334566,32.822086,31.738683,25.34005,30.137845,33.739342,28.646833,23.462532,33.9556,24.509254,33.939738,23.538789,30.7,28.599762,26.219771,26.722338,24.082022,27.136515,28.070175
2,openai_whisper-large-v3-turbo__vimd,label_text_normalized_toy,test_metrics.txt,0.167582,NaN,NaN,NaN,NaN,NaN,NaN,26.879924,NaN,NaN,NaN,NaN,NaN,29.289173,21.513867,26.511628,26.041144,33.544612,28.89991,24.453941,23.904382,26.736943,25.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
